In [318]:
import torch
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd

In [ ]:
test_results = pd.read_csv('config.csv', dtype=str)
# to dataframe
test_results = pd.DataFrame(test_results)
# Strip any leading or trailing spaces from the column names
test_results.columns = test_results.columns.str.strip()
test_results = test_results.apply(lambda x: x.str.strip() if x.dtype == "object" else str(x))
file_list = []
for i in range(len(test_results)):
    seeds = str(test_results.iloc[i]['seed']).split(' ')
    dataset = test_results.iloc[i]['dataset']
    usage = test_results.iloc[i]['usage']
    model = test_results.iloc[i]['model']
    lr = test_results.iloc[i]['lr']
    snr = test_results.iloc[i]['snr']
    snvr = test_results.iloc[i]['snvr']
    weight_decay = test_results.iloc[i]['weight_decay']
    optim = test_results.iloc[i]['optim']
    
    valid_top1_acc_list = []
    valid_top5_acc_list = []
    test_top1_acc_list = []
    test_top5_acc_list = []

    ckpt_path = f'./ckpt/{dataset}_{usage}_{model}_{lr}_{snr}_{snvr}_{weight_decay}_{optim}.pth'
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path)
        file_list.append(ckpt_path.split('/')[-1])
    
        for seed in ckpt.keys():
            valid_top1_acc_list.append(ckpt[seed]['best_epoch']['valid_top1_acc']*100)
            valid_top5_acc_list.append(ckpt[seed]['best_epoch']['valid_top5_acc']*100)
            test_top1_acc_list.append(ckpt[seed]['best_epoch']['test_top1_acc']*100)
            test_top5_acc_list.append(ckpt[seed]['best_epoch']['test_top5_acc']*100)
    
    test_results.at[i, 'seed'] = f'{len(valid_top1_acc_list)}/{len(seeds)}'
    if len(valid_top1_acc_list) == 0: continue
    test_results.at[i, 'valid_acc@1'] = f'{np.mean(valid_top1_acc_list):.2f} ± {np.std(valid_top1_acc_list):.2f}'
    test_results.at[i, 'valid_acc@5'] = f'{np.mean(valid_top5_acc_list):.2f} ± {np.std(valid_top5_acc_list):.2f}'
    test_results.at[i, 'test_acc@1'] = f'{np.mean(test_top1_acc_list):.2f} ± {np.std(test_top1_acc_list):.2f}'
    test_results.at[i, 'test_acc@5'] = f'{np.mean(test_top5_acc_list):.2f} ± {np.std(test_top5_acc_list):.2f}'

test_results = test_results.drop(columns=['epochs','warm_up','batch_size'])
print(test_results.to_string())

   seed       dataset usage              model    lr   snr  snvr weight_decay  optim     prefix   valid_acc@1   valid_acc@5    test_acc@1    test_acc@5
0   1/1      imagenet  0.05  convnext_tiny-0.5  4e-3   0.0   0.0         0.05  adamw             27.21 ± 0.00  46.22 ± 0.00  24.87 ± 0.00  43.84 ± 0.00
1   0/1      imagenet  0.05  convnext_tiny-0.5  4e-3  1e-3   0.0          0.0  adamw  decouple_           NaN           NaN           NaN           NaN
2   1/1      imagenet  0.05  convnext_tiny-0.5  4e-3  1e-5   0.0         0.05  adamw             27.80 ± 0.00  46.41 ± 0.00  25.18 ± 0.00  44.39 ± 0.00
3   0/1      imagenet  0.05  convnext_tiny-0.5  4e-3  1e-6   0.0         0.05  adamw                      NaN           NaN           NaN           NaN
4   0/1      imagenet  0.05  convnext_tiny-0.5  4e-3  1e-6   0.1         0.05  adamw                      NaN           NaN           NaN           NaN
5   1/1  imagenet-112  0.05  convnext_tiny-0.5  4e-3   0.0   0.0         5e-4   adam    

In [320]:
for file in os.listdir('./ckpt/'):
    if file in file_list:
        print(file)
    # if file not in file_list:
    #     os.remove(f'./ckpt/{file}')

imagenet-112_0.05_convnext_tiny_4e-3_1e-5_1e-3_0.0_adam.pth
imagenet-112_0.05_swin-v2_tiny-0.5_1e-4_1e-6_0.0_0.0_adam.pth
imagenet-112_0.05_swin-v2_tiny-0.5_1e-4_1e-5_0.0_0.0_adam.pth
imagenet_0.05_convnext_tiny-0.5_4e-3_0.0_0.0_0.05_adamw.pth
imagenet-112_0.05_swin-v2_tiny-0.5_1e-4_1e-3_0.0_0.0_adam.pth
imagenet-112_0.1_convnext_tiny-0.5_4e-3_0.0_0.0_0.05_adamw.pth
imagenet-112_0.05_swin-v2_tiny-0.5_1e-4_0.0_0.0_1e-3_adam.pth
imagenet-112_0.05_convnext_tiny_4e-3_0.0_0.0_1e-5_adam.pth
imagenet-112_0.1_convnext_tiny-0.5_4e-3_1e-6_0.1_0.05_adamw.pth
imagenet-112_0.05_convnext_tiny_4e-3_1e-6_1e-5_0.0_adam.pth
imagenet-112_0.05_convnext_tiny-0.5_4e-3_1e-6_0.0_5e-4_adam.pth
imagenet-112_0.05_convnext_tiny_4e-3_1e-5_0.01_0.0_adam.pth
imagenet-112_0.1_convnext_tiny-0.5_4e-3_1e-6_1e-3_0.05_adamw.pth
imagenet-112_0.05_swin-v2_tiny-0.5_1e-3_0.0_0.0_0.0_adam.pth
imagenet-112_0.1_convnext_tiny-0.5_4e-3_1e-4_0.0_0.05_adamw.pth
imagenet-112_0.05_swin-v2_tiny-0.5_1e-4_1e-4_1.0_0.0_adam.pth
imagenet-1

In [321]:
import torch.nn as nn
from torchvision.ops.misc import Permute

block = nn.Sequential(
    nn.Conv2d(96, 96, kernel_size=7, padding=3, groups=96, bias=True),
    Permute([0, 2, 3, 1]),
    nn.LayerNorm(96),
    nn.Linear(in_features=96, out_features=4 * 96, bias=True),
    nn.GELU(),
    nn.Linear(in_features=4 * 96, out_features=96, bias=True),
    Permute([0, 3, 1, 2]),
)

x = torch.randn(4, 96, 224, 112)
y = block(x)
print(y.shape)

torch.Size([4, 96, 224, 112])
